[![../img/pythonista.png](../img/pythonista.png)](https://www.pythonista.io)

# *NumPy* esencial para el análisis de datos.

**py401 · EDA Práctico** · Pythonista® · Instructor: José Luis Chiquete · Módulo I: *Datos y su calidad*

> 🔑 **Versión con soluciones.** Solo para el instructor o para revisión posterior.

En este *notebook* se repasarán los fundamentos de *NumPy* (`dtype`, operaciones vectorizadas, *broadcasting* y máscaras booleanas) y cuatro decisiones con *NumPy* que cambian los resultados de un análisis.

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* **Ejercicio básico:** lo que todos deben dominar.
* **Reto (opcional):** para quien termina antes o quiere profundizar. Si no da tiempo en clase, se puede resolver como práctica.
* Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.

## 📚 Referencia de la API: curso py311.

Este curso no repasa la API método por método: esa explicación detallada está en los *notebooks* del curso [py311](https://github.com/PythonistaMX/py311). Aquí se hace un repaso breve y el tiempo se dedica a las **decisiones de análisis**: qué tipo de dato elegir, qué hacer con un faltante y qué consecuencias tiene cada decisión. Para este *notebook*:

| Tema | *Notebooks* de py311 |
|---|---|
| *NumPy* | [02 Conceptos básicos de *NumPy*](https://github.com/PythonistaMX/py311/blob/main/02_conceptos_basicos_de_numpy.ipynb) · [03 Gestión de arreglos](https://github.com/PythonistaMX/py311/blob/main/03_gestion_de_arreglos_de_numpy.ipynb) · [04 Arreglos con contenido aleatorio](https://github.com/PythonistaMX/py311/blob/main/04_arreglos_con_contenido_aleatorio.ipynb) · [05 Operaciones básicas con arreglos](https://github.com/PythonistaMX/py311/blob/main/05_operaciones_basicas_con_arreglos.ipynb) |
| *pandas* | [09 Introducción a *pandas*](https://github.com/PythonistaMX/py311/blob/main/09_introduccion_a_pandas.ipynb) · [10 Tipos de datos](https://github.com/PythonistaMX/py311/blob/main/10_tipos_de_datos_de_pandas.ipynb) · [11 Operaciones básicas con *dataframes*](https://github.com/PythonistaMX/py311/blob/main/11_operaciones_basicas_con_dataframes.ipynb) · [12 Índices y multiíndices](https://github.com/PythonistaMX/py311/blob/main/12_indices_y_multiindices.ipynb) · [13 Datos categóricos](https://github.com/PythonistaMX/py311/blob/main/13_datos_categoricos.ipynb) |

> ⚠️ **Nota: py311 usa *pandas* 2.** Sus *notebooks* se escribieron para *pandas* 2.x y este curso usa *pandas* 3, así que algunos resultados se verán distintos. Por ejemplo, allá el texto aparece con `dtype` `object` (aquí es `str`), se explica el parámetro `inplace` y la asignación encadenada como si modificaran el original (con *pandas* 3 ya no lo hacen sobre columnas o selecciones) y puede aparecer `SettingWithCopyWarning`, que ya no existe. La sección 1.1 de `03_pandas_3_esencial.ipynb` explica estos cambios. Cuando haya diferencias, se sigue lo que muestra este *notebook*.

### Preparación del entorno.

Se importarán las bibliotecas con sus alias convencionales, se ubicará la carpeta `data/` (que ya trae los CSV del curso, sin necesidad de internet) y se fijará la semilla `401` para que los resultados aleatorios sean reproducibles.

* La siguiente celda importará las bibliotecas y definirá `DATA` y `rng`.

In [1]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = next(os.path.join(p, "data") for p in (os.curdir, os.pardir)
            if os.path.isfile(os.path.join(p, "data", "penguins.csv")))
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(401)

* La siguiente celda mostrará las versiones instaladas. El curso usa *pandas* 3.

In [2]:
{"NumPy": np.__version__, "pandas": pd.__version__, "seaborn": sns.__version__}

{'NumPy': '2.5.4', 'pandas': '3.0.6', 'seaborn': '0.13.2'}

---
## 1. *NumPy* esencial.

[*NumPy*](https://numpy.org) es la base de casi todo el ecosistema científico de *Python*: *pandas*, *SciPy*, *statsmodels* y *Matplotlib* lo usan por dentro. Su objeto central es el **arreglo** (`ndarray`):

* **Homogéneo:** todos los elementos tienen el mismo `dtype` (`int64`, `float64`, `bool`, …).
* **De tamaño fijo y contiguo en memoria**, lo que permite operaciones **vectorizadas** escritas en C.

### 1.1 Repaso exprés.

| Concepto | Lo esencial | En py311 |
|---|---|---|
| Arreglo y `dtype` | Homogéneo; mezclar enteros y flotantes promueve a `float64`; `np.nan` es flotante | [02](https://github.com/PythonistaMX/py311/blob/main/02_conceptos_basicos_de_numpy.ipynb), [03](https://github.com/PythonistaMX/py311/blob/main/03_gestion_de_arreglos_de_numpy.ipynb) |
| Aleatoriedad reproducible | `rng = np.random.default_rng(401)` y sus métodos (`rng.normal`, `rng.choice`, …) | [04](https://github.com/PythonistaMX/py311/blob/main/04_arreglos_con_contenido_aleatorio.ipynb) |
| Vectorización | Operar sobre todo el arreglo sin bucles: más claro y mucho más rápido | [05](https://github.com/PythonistaMX/py311/blob/main/05_operaciones_basicas_con_arreglos.ipynb) |
| *Broadcasting* | Formas compatibles si, de derecha a izquierda, cada dimensión es igual o vale 1 | [05](https://github.com/PythonistaMX/py311/blob/main/05_operaciones_basicas_con_arreglos.ipynb) |
| Máscaras booleanas | Una comparación da `True`/`False`; sirve para filtrar y contar (`True` vale 1); se combinan con `&`, `\|` y `~` | [05](https://github.com/PythonistaMX/py311/blob/main/05_operaciones_basicas_con_arreglos.ipynb) |

**Concepto: arreglo y `dtype`.** Un arreglo de *NumPy* guarda $n$ valores del **mismo tipo** en un bloque continuo de memoria. Esa homogeneidad es la que permite operar sobre todos a la vez. El `dtype` dice cómo se interpreta cada valor (`int64`: entero de 64 bits; `float64`: número de punto flotante). En un EDA importa porque el tipo condiciona qué valores caben: un entero no puede guardar un faltante (`np.nan` es un flotante).

* La siguiente celda creará el arreglo `masas` (g) de 8 pingüinos y mostrará su `dtype` y la promoción a `float64` cuando hay un `NaN`.

In [3]:
masas = np.array([3750, 3800, 3250, 3450, 3650, 3625, 4675, 3475])
masas.dtype, masas.shape, np.array([3750, np.nan, 3250]).dtype

(dtype('int64'), (8,), dtype('float64'))

**Interpretación:** las 8 masas son enteros (`int64`) con forma `(8,)`, un arreglo de una dimensión. Al agregar un `np.nan`, *NumPy* promueve todo el arreglo a `float64`. Por eso, en *pandas*, una columna de enteros con faltantes aparece como `float64` (lo verás con `body_mass_g`) y por eso existen los tipos *nullable* como `Int64` (sección 1.6 de `04_tipos_de_variables.ipynb`).

**Concepto: vectorización y máscaras booleanas.** Una operación **vectorizada** (`masas / 1000`) se aplica a todos los elementos sin escribir un bucle. Una comparación (`masas > 3600`) produce una **máscara**: un arreglo de `True`/`False` del mismo tamaño. Como `True` vale 1 y `False` vale 0, la suma de una máscara **cuenta** y su media es una **proporción**:

$$\hat{p} = \frac{1}{n}\sum_{i=1}^{n} \mathbb{1}(x_i > 3600)$$

Este patrón ("contar con una máscara") se usará en todo el curso para calcular proporciones de faltantes, de atípicos o de éxitos.

* La siguiente celda repasará vectorización y máscaras: kilogramos, cuántos superan 3,600 g, en qué proporción y una etiqueta con `np.where`.

In [4]:
pesados = masas > 3600
masas / 1000, pesados.sum(), pesados.mean(), np.where(pesados, "pesado", "ligero")

(array([3.75 , 3.8  , 3.25 , 3.45 , 3.65 , 3.625, 4.675, 3.475]),
 np.int64(5),
 np.float64(0.625),
 array(['pesado', 'pesado', 'ligero', 'ligero', 'pesado', 'pesado',
        'pesado', 'ligero'], dtype='<U6'))

**Interpretación:** 5 de los 8 pingüinos superan 3,600 g, es decir, una proporción de 0.625. `np.where` asigna una etiqueta a cada elemento según la máscara: así se crean variables categóricas derivadas a partir de una numérica.

**Concepto: *broadcasting*.** Para restar la media de cada columna a cada fila no hace falta repetir el vector de medias: *NumPy* lo "estira" virtualmente. Las formas `(4, 3)` y `(3,)` son compatibles porque, comparando de derecha a izquierda, las dimensiones son iguales (3 y 3). Centrar ($x - \bar{x}$) y estandarizar ($z = (x - \bar{x}) / s$) son pasos frecuentes en un EDA para comparar variables en escalas distintas.

* La siguiente celda creará la matriz `medidas` (4 pingüinos × largo del pico, profundidad del pico y aleta, en mm) y la centrará por columna con *broadcasting*: `(4, 3) - (3,)`.

In [5]:
medidas = np.array([
    [39.1, 18.7, 181],
    [39.5, 17.4, 186],
    [46.5, 17.9, 192],
    [50.0, 15.2, 218],
])
(medidas - medidas.mean(axis=0)).round(2)

array([[ -4.67,   1.4 , -13.25],
       [ -4.27,   0.1 ,  -8.25],
       [  2.73,   0.6 ,  -2.25],
       [  6.23,  -2.1 ,  23.75]])

**Interpretación:** cada columna centrada tiene media 0. Un valor positivo indica que ese pingüino está por encima del promedio en esa medida (el cuarto pingüino tiene la aleta 23.75 mm por encima). Centrar no cambia la forma de los datos, solo su origen.

### 1.2 Decisiones con *NumPy* que cambian resultados.

El código "corre" en los cuatro casos siguientes; lo que cambia es **el número que se reporta**. Son decisiones que hay que tomar a propósito.

**Decisión 1: ¿desviación estándar muestral o poblacional?** *NumPy* usa por defecto `ddof=0` (poblacional) y *pandas* `ddof=1` (muestral). Con los mismos datos se obtienen números distintos. Como en EDA casi siempre se trabaja con una **muestra**, se usa `ddof=1`.

**Concepto: desviación estándar poblacional y muestral.** Con $n$ datos,

$$s_{\text{pobl}} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(x_i - \bar{x})^2} \qquad s = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(x_i - \bar{x})^2}$$

Dividir entre $n - 1$ (`ddof=1`) corrige que $\bar{x}$ se calculó con los mismos datos y tiende a quedar "demasiado cerca" de ellos. Cuando los datos son una **muestra**, como casi siempre en un EDA, se usa $s$.

* La siguiente celda comparará la desviación estándar de `masas` con *NumPy* por defecto, con `ddof=1` y con *pandas*.

In [6]:
{"NumPy por defecto (ddof=0)": round(float(masas.std()), 1),
 "NumPy con ddof=1": round(float(masas.std(ddof=1)), 1),
 "pandas por defecto (ddof=1)": round(float(pd.Series(masas).std()), 1)}

{'NumPy por defecto (ddof=0)': 401.0,
 'NumPy con ddof=1': 428.6,
 'pandas por defecto (ddof=1)': 428.6}

**Interpretación:** con 8 datos, la diferencia es de casi 28 g (401.0 contra 428.6). Con *n* grande la diferencia se vuelve despreciable, pero con muestras pequeñas no. **Error común que se evita:** mezclar resultados de *NumPy* y de *pandas* en un mismo informe sin notar que usan `ddof` distinto.

**Decisión 2: ¿qué hacer con `NaN` al resumir?** En *NumPy*, un solo `NaN` convierte la media en `nan` (`np.nanmean` lo ignora); en *pandas* los faltantes se ignoran **en silencio**. Ninguna opción es neutral: ignorar faltantes cambia el denominador y, si los faltantes no son aleatorios, sesga el resultado (notebook `08_valores_faltantes.ipynb`). Se resuelve en el ejercicio 1.

📖 **Lectura opcional.** **Decisión 3: ¿un `dtype` pequeño para ahorrar memoria?** Los enteros de tamaño fijo **se desbordan sin avisar** en operaciones con arreglos. Ahorrar memoria con `uint8` o `int16` solo es seguro si se conoce el rango de los datos.

* La siguiente celda sumará 10 a un arreglo `uint8` (rango 0 a 255) y mostrará el desbordamiento silencioso.

In [7]:
conteos = np.array([250, 120], dtype=np.uint8)
conteos + np.array([10, 10], dtype=np.uint8)

array([  4, 130], dtype=uint8)

**Interpretación:** 250 + 10 da 4 (no 260) porque `uint8` solo representa de 0 a 255 y "da la vuelta". No hay error ni advertencia: el resultado incorrecto pasaría a cualquier análisis posterior.

📖 **Lectura opcional.** **Decisión 4: ¿comparar flotantes con `==`?** Los flotantes son aproximaciones binarias. Un filtro con `==` sobre valores calculados puede dejar fuera filas que "deberían" entrar; se usa `np.isclose` con una tolerancia.

* La siguiente celda comparará `0.1 + 0.2` con `0.3` usando `==` y `np.isclose`.

In [8]:
0.1 + 0.2 == 0.3, np.isclose(0.1 + 0.2, 0.3)

(False, np.True_)

**Interpretación:** `0.1 + 0.2 == 0.3` es `False` porque ninguno de esos números tiene representación binaria exacta; `np.isclose` compara con una tolerancia y da `True`. Al filtrar valores calculados (por ejemplo, proporciones), se usa `np.isclose` o se redondea antes.

### Ejercicio 1 · Máscaras y faltantes.

Con el arreglo `masa_g` (la masa de **todos** los pingüinos del *dataset*, incluidos los `NaN`):
1. Conviértelo a kilogramos en `masa_kg`.
2. Calcula la media **ignorando faltantes** (pista: `np.nanmean`). ¿Qué pasa si usas `.mean()`?
3. Cuenta cuántos pingüinos pesan más de 4.5 kg y qué proporción representan **entre los que tienen dato**.

**🎯 Objetivo:** practicar operaciones vectorizadas y máscaras sobre datos reales con faltantes, y notar cómo cambia el **denominador** de una proporción según se traten los `NaN`. **Qué se espera aprender:** que un faltante no es un detalle técnico, sino una decisión que cambia los números reportados.

* La siguiente celda extraerá la columna `body_mass_g` como arreglo de *NumPy*.

In [9]:
masa_g = pd.read_csv(os.path.join(DATA, "penguins.csv"))["body_mass_g"].to_numpy()
masa_g[:6], masa_g.dtype, masa_g.shape

(array([3750., 3800., 3250.,   nan, 3450., 3650.]), dtype('float64'), (344,))

* La siguiente celda muestra una solución posible.

In [10]:
masa_kg = masa_g / 1000
con_dato = ~np.isnan(masa_kg)
mas_de_45 = masa_kg[con_dato] > 4.5
{
    "media con .mean()": masa_kg.mean(),
    "media con np.nanmean()": np.nanmean(masa_kg).round(3),
    "más de 4.5 kg": int(mas_de_45.sum()),
    "proporción entre los que tienen dato": mas_de_45.mean().round(3),
}

{'media con .mean()': np.float64(nan),
 'media con np.nanmean()': np.float64(4.202),
 'más de 4.5 kg': 115,
 'proporción entre los que tienen dato': np.float64(0.336)}

**✅ Respuesta sugerida:**

`.mean()` devuelve `nan`: un solo faltante contamina el resultado. `np.nanmean()` da ≈ 4.202 kg. Hay 115 pingüinos de más de 4.5 kg, el 33.6% de los 342 con dato. Si se dividiera entre los 344 (incluidos los `NaN`) se subestimaría la proporción.

### Ejercicio 2 · Estandarizar con *broadcasting*.

Estandariza cada **columna** de `medidas` (puntaje *z*: restar la media y dividir entre la desviación estándar de su columna) **sin bucles**. Verifica que las medias resultantes sean ≈ 0 y las desviaciones ≈ 1.

**🎯 Objetivo:** aplicar *broadcasting* para estandarizar, $z_{ij} = (x_{ij} - \bar{x}_j) / s_j$, sin bucles. **Qué se espera aprender:** que un puntaje *z* expresa cada valor en "desviaciones estándar respecto a la media de su columna", lo que permite comparar medidas en unidades distintas (mm y g).

* La siguiente celda muestra una solución posible.

In [11]:
z = (medidas - medidas.mean(axis=0)) / medidas.std(axis=0, ddof=1)
z.round(3), z.mean(axis=0).round(10), z.std(axis=0, ddof=1).round(10)

(array([[-0.872,  0.934, -0.805],
        [-0.797,  0.067, -0.501],
        [ 0.508,  0.4  , -0.137],
        [ 1.161, -1.401,  1.443]]),
 array([0., 0., 0.]),
 array([1., 1., 1.]))

### Reto 1 · Matriz de distancias sin bucles.

Calcula la matriz 4×4 de **distancias euclidianas** entre todos los pares de pingüinos de `z` usando solo *broadcasting* (pista: `z[:, None, :] - z[None, :, :]` tiene forma `(4, 4, 3)`). ¿Qué par de pingüinos es el más parecido?

**🎯 Objetivo:** llevar el *broadcasting* a tres dimensiones para calcular $d(a, b) = \sqrt{\sum_j (z_{aj} - z_{bj})^2}$ entre todos los pares. **Qué se espera aprender:** cómo se mide la "semejanza" entre observaciones, idea base del agrupamiento.

* La siguiente celda muestra una solución posible.

In [12]:
diferencias = z[:, None, :] - z[None, :, :]
distancias = np.sqrt((diferencias ** 2).sum(axis=-1))
sin_diagonal = distancias.copy()
np.fill_diagonal(sin_diagonal, np.inf)
par = np.unravel_index(sin_diagonal.argmin(), sin_diagonal.shape)
distancias.round(2), par

(array([[0.  , 0.92, 1.62, 3.83],
        [0.92, 0.  , 1.4 , 3.13],
        [1.62, 1.4 , 0.  , 2.48],
        [3.83, 3.13, 2.48, 0.  ]]),
 (np.int64(0), np.int64(1)))

**💡 Ideas clave de la sección 1:**
* Un arreglo es homogéneo; un `NaN` fuerza `float64`.
* La vectorización y las máscaras permiten contar y calcular proporciones sin bucles.
* El *broadcasting* centra y estandariza columnas en una línea.
* Decisiones que cambian números: `ddof`, el tratamiento de `NaN`, el tamaño del `dtype` y la comparación de flotantes.

---
## ✅ Cierre.

**Práctica (en Classroom):** terminar los ejercicios pendientes de este *notebook* y, si es posible, un reto.

**Siguiente *notebook*:** `03_pandas_3_esencial.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>